In [5]:
import pandas as pd

import sys
sys.path.insert(1, '../../scripts')
from utils.load_environmental_variables import build_files_path, input_data_path
prebuild = '/data2/hratch/human_me/prebuild/'

Format

In [7]:
ptr = pd.read_csv(prebuild + 'PTR_Gagneur_unprocessed.tsv', sep = '\t')
ptr.replace('   NA', float('nan'), inplace = True)
ptr.replace('    NA', float('nan'), inplace = True)

ptr_cols = [col for col in ptr.columns if 'PTR' in col]
for col in ptr_cols:
    ptr[col] = ptr[col].astype(float)

ptr = pd.concat([ptr[['EnsemblGeneID']], 10**ptr[ptr_cols]], axis = 1)

Map ID

In [10]:
ehm = pd.read_csv(prebuild + 'sequence_information/identifiers.txt', sep = '\t')
ehm = ehm.loc[ehm['Ensembl gene ID'].dropna().index,:]
ehm = ehm.loc[ehm['HGNC ID'].dropna().index,:]

In [11]:
# # get many to one mappings
# he = dict()
# for i in ehm.index:
#     ensg = ehm.loc[i, 'Ensembl gene ID']
#     hgnc = ehm.loc[i, 'HGNC ID']
#     if hgnc in he.keys():
#         he[hgnc] += [ensg]
#     else:
#         he[hgnc] = [ensg]
# he = {k: list(set(v)) for k,v in he.items()}
# he = {k:v for k,v in he.items() if len(v)>1}
# if len(he) > 0:
#     raise ValueError('Have not dealt with one to many mappings')
eh = dict()
for i in ehm.index:
    ensg = ehm.loc[i, 'Ensembl gene ID']
    hgnc = ehm.loc[i, 'HGNC ID']
    if ensg in eh.keys():
        eh[ensg] += [hgnc]
    else:
        eh[ensg] = [hgnc]
eh = {k: list(set(v)) for k,v in eh.items()}
eh = {k:v[0] for k,v in eh.items() if len(v)==1}
if len(list(eh.values())) != len(set(eh.values())):
    raise ValueError('Have not dealt with many to one mappings')


# get one to many mappings
eh = dict()
for i in ehm.index:
    ensg = ehm.loc[i, 'Ensembl gene ID']
    hgnc = ehm.loc[i, 'HGNC ID']
    if ensg in eh.keys():
        eh[ensg] += [hgnc]
    else:
        eh[ensg] = [hgnc]
eh = {k: list(set(v)) for k,v in eh.items()}
eh = {k:v for k,v in eh.items() if len(v)>1}



In [12]:
# do the mapping
eh_ = dict(zip(ehm['Ensembl gene ID'], ehm['HGNC ID']))
eh_ = {k: v for k,v in eh_.items() if k not in eh.keys()}
for k,v in eh.items():
    eh_[k] = v

ptr['HGNC ID'] = ptr['EnsemblGeneID'].map(eh_)
if ptr[ptr.EnsemblGeneID.isin(list(eh.keys()))].shape[0] > 1:
    raise ValueError('Deal with the many to one mappings')
ptr = ptr.loc[ptr['HGNC ID'].dropna().index,:]



more formatting

In [13]:
ptr.index = ptr['HGNC ID']
ptr.drop(columns = ['HGNC ID', 'EnsemblGeneID'], inplace = True)
ptr.columns = ['_'.join(col.split('_')[:-1]) for col in ptr.columns]
ptr['Median'] = ptr.median(axis = 1)
ptr.to_csv(build_files_path + 'PTR_Gagneur_processed.tsv', sep = '\t')